# 08 - Cross-Experiment Comparison

Aggregates all result artefacts written and produces the final comparison tables, plots, and interpretation notes referenced in the paper

In [ ]:
import sys
from pathlib import Path

#sys.path.insert(0, str(Path.cwd().parent if (Path.cwd() / "notebooks").exists() else Path.cwd()))
current_notebook_dir = Path.cwd()

if current_notebook_dir.name == "notebooks":
    project_root = current_notebook_dir.parent
else:
    project_root = current_notebook_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

from src.utils import load_config, get_logger
from data.paths import get_dataset_config, get_results_dir, get_split_file
from src.classes import get_class_registry

In [ ]:
DATASET = "ahn3"

config = load_config("default")
logger = get_logger("08_compare_results")
dataset_cfg = get_dataset_config(DATASET)
class_registry = get_class_registry(DATASET)
class_names = class_registry.class_names_in_order()

MODELS = ["baseline", "uncertainty_feature", "uncertainty_loss"]
MODEL_LABELS = {
    "baseline":             "Baseline ANN",
    "uncertainty_feature":  "ANN + Uncertainty Feature",
    "uncertainty_loss":     "ANN + Uncertainty Loss",
}
COLORS = {
    "baseline":             "#4C72B0",
    "uncertainty_feature":  "#DD8452",
    "uncertainty_loss":     "#55A868",
}

## Load all per-model results
Reads `summary.csv`, `per_class_iou.csv`, and `pavpu.csv` from each model's results directory

In [ ]:
summaries    = {}
per_class    = {}
pavpu_detail = {}
missing      = []

for model in MODELS:
    res_dir = get_results_dir(DATASET, model)
    s_path  = res_dir / "summary.csv"
    p_path  = res_dir / "per_class_iou.csv"
    pv_path = res_dir / "pavpu.csv"

    if not (s_path.exists() and p_path.exists() and pv_path.exists()):
        missing.append(model)
        logger.warning(f"Results missing for '{model}' - skipping. "
                       f"Run notebook 04/05/06 first.")
        continue

    summaries[model]    = pd.read_csv(s_path)
    per_class[model]    = pd.read_csv(p_path)
    pavpu_detail[model] = pd.read_csv(pv_path)

available = list(summaries.keys())
print(f"Loaded: {available}")
if missing:
    print(f"Missing (skipped): {missing}")

## Summary table - accuracy, mIoU, PAvPU
- Accuracy and mIoU measure correctness (on spatially held-out test)
- PAvPU measures confidence and correctness

In [ ]:
rows = []
for model in available:
    s = summaries[model].iloc[0]
    pv = pavpu_detail[model].iloc[0]
    rows.append({
        "Model": MODEL_LABELS[model],
        "Accuracy": s["accuracy"],
        "mIoU": s["mIoU"],
        "PAvPU": pv["PAvPU"],
        "zero_IoU_classes": int(s["zero_iou_classes"]),
        "u_threshold": pv["uncertainty_threshold"],
    })

summary_df = pd.DataFrame(rows).set_index("Model")
print(summary_df.to_string())

In [ ]:
if "baseline" in available:
    base = summary_df.loc[MODEL_LABELS["baseline"]]
    delta_df = summary_df[["Accuracy","mIoU","PAvPU"]].subtract(
        base[["Accuracy","mIoU","PAvPU"]]
    ).rename(columns=lambda c: f"Δ{c}")
    comparison = pd.concat([summary_df[["Accuracy","mIoU","PAvPU"]], delta_df], axis=1)
    print("\nVs. baseline:")
    print(comparison.to_string())

## PAvPU breakdown - AC / AU / IC / IU

PAvPU - Confidence vs Correctness - Idea

| | Certain | Uncertain |
|---|---|---|
| **Accurate** | AC (good) | AU (wasted signal) |
| **Inaccurate** | IC (bad) | IU (good) |

A good uncertainty-aware model should push points from IC to IU (the model becomes uncertain when it's wrong) and from AU to AC (the model
becomes confident when it's right). If neither shift happens relative to baseline, the uncertainty signal isn't influencing the model usefully

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(available))
bars = axes[0].bar(
    x,
    [pavpu_detail[m].iloc[0]["PAvPU"] for m in available],
    color=[COLORS[m] for m in available],
    width=0.5,
)
axes[0].set_xticks(x)
axes[0].set_xticklabels([MODEL_LABELS[m] for m in available], rotation=12, ha="right")
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("PAvPU")
axes[0].set_title("PAvPU score")
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.01, f"{h:.3f}",
                 ha="center", va="bottom", fontsize=10, fontweight="bold")
axes[0].axhline(0.5, color="grey", linestyle="--", linewidth=0.8, label="random baseline (0.5)")
axes[0].legend(fontsize=8)

cats   = ["N_AC", "N_AU", "N_IC", "N_IU"]
clrs   = ["#2ecc71", "#f1c40f", "#e74c3c", "#3498db"]
lbls   = ["AC (accurate & certain)", "AU (accurate & uncertain)",
          "IC (inaccurate & certain)", "IU (inaccurate & uncertain)"]
bottoms = np.zeros(len(available))

for cat, color, label in zip(cats, clrs, lbls):
    vals = np.array([pavpu_detail[m].iloc[0][cat] for m in available], dtype=float)
    totals = np.array([
        sum(pavpu_detail[m].iloc[0][c] for c in cats)
        for m in available
    ], dtype=float)
    pcts = vals / np.where(totals > 0, totals, 1)
    axes[1].bar(x, pcts, bottom=bottoms, color=color, label=label, width=0.5)
    for i, (p, b) in enumerate(zip(pcts, bottoms)):
        if p > 0.04:
            axes[1].text(x[i], b + p/2, f"{p*100:.0f}%",
                         ha="center", va="center", fontsize=8, color="white", fontweight="bold")
    bottoms += pcts

axes[1].set_xticks(x)
axes[1].set_xticklabels([MODEL_LABELS[m] for m in available], rotation=12, ha="right")
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("Fraction of patches")
axes[1].set_title("Patch category breakdown")
axes[1].legend(fontsize=8, loc="lower right")

plt.tight_layout()
out = get_results_dir(DATASET, "comparison") / "pavpu_breakdown.png"
plt.savefig(str(out), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved at : {out}")

## Per-class IoU - heatmap and delta vs baseline
- The heatmap shows raw IoU per class per model. The delta plot shows where each uncertainty-aware model **gains or loses** relative to the baseline
- Ideally uncertainty should help on geometrically ambiguous or minority classes, so look here first for evidence of that

In [ ]:
iou_df = pd.DataFrame({
    MODEL_LABELS[m]: per_class[m].set_index("class_name")["IoU"]
    for m in available
})

fig, axes = plt.subplots(1, 2, figsize=(14, max(5, len(iou_df) * 0.55)))

# Raw IoU heatmap
im = axes[0].imshow(iou_df.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
axes[0].set_xticks(range(len(iou_df.columns)))
axes[0].set_xticklabels(iou_df.columns, rotation=15, ha="right", fontsize=8)
axes[0].set_yticks(range(len(iou_df.index)))
axes[0].set_yticklabels(iou_df.index, fontsize=8)
plt.colorbar(im, ax=axes[0], label="IoU")
axes[0].set_title("Per-class IoU")
for i in range(len(iou_df.index)):
    for j in range(len(iou_df.columns)):
        axes[0].text(j, i, f"{iou_df.values[i,j]:.2f}",
                     ha="center", va="center", fontsize=7,
                     color="black" if iou_df.values[i,j] < 0.7 else "white")

# Delta vs baseline
if "baseline" in available and len(available) > 1:
    delta_df = iou_df.subtract(iou_df[MODEL_LABELS["baseline"]], axis=0).drop(
        columns=MODEL_LABELS["baseline"]
    )
    vabs = max(abs(delta_df.values.min()), abs(delta_df.values.max()), 0.05)
    norm = TwoSlopeNorm(vmin=-vabs, vcenter=0, vmax=vabs)
    im2 = axes[1].imshow(delta_df.values, aspect="auto", cmap="RdYlGn", norm=norm)
    axes[1].set_xticks(range(len(delta_df.columns)))
    axes[1].set_xticklabels(delta_df.columns, rotation=15, ha="right", fontsize=8)
    axes[1].set_yticks(range(len(delta_df.index)))
    axes[1].set_yticklabels(delta_df.index, fontsize=8)
    plt.colorbar(im2, ax=axes[1], label="ΔIoU vs baseline")
    axes[1].set_title("ΔIoU vs baseline  (green = improvement)")
    for i in range(len(delta_df.index)):
        for j in range(len(delta_df.columns)):
            axes[1].text(j, i, f"{delta_df.values[i,j]:+.2f}",
                         ha="center", va="center", fontsize=7)
else:
    axes[1].set_visible(False)

plt.tight_layout()
out = get_results_dir(DATASET, "comparison") / "per_class_iou_heatmap.png"
plt.savefig(str(out), dpi=150, bbox_inches="tight")
plt.show()

## Uncertainty distribution - train vs test
Ideally,
- **Test uncertainty should be higher on average than train** GDA was fit on train, so train points are closer to class centroids by construction)
- **Per-class uncertainty** on test should be higher for geometrically ambiguous classes. If not, the GDA feature signal is weak

In [ ]:
train_df = pd.read_csv(get_split_file(DATASET, "train"))
test_df  = pd.read_csv(get_split_file(DATASET, "test"))

if "__u_norm__" not in train_df.columns:
    print("   __u_norm__ not found in split CSVs.")
    print("   Run notebook 03 first to compute and attach uncertainty.")
else:
    u_train = train_df["__u_norm__"].values
    u_test  = test_df["__u_norm__"].values

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Overall train vs test
    axes[0].hist(np.clip(u_train, 0, 2), bins=50, alpha=0.65,
                 label=f"Train (mean={u_train.mean():.3f})", color="#4C72B0", density=True)
    axes[0].hist(np.clip(u_test,  0, 2), bins=50, alpha=0.65,
                 label=f"Test  (mean={u_test.mean():.3f})",  color="#DD8452", density=True)
    axes[0].set_xlabel("Normalised epistemic uncertainty (clipped at 2 for display)")
    axes[0].set_ylabel("Density")
    axes[0].set_title("Train vs test uncertainty distribution")
    axes[0].legend()

    # Per-class on test
    for idx, name in enumerate(class_names):
        mask = test_df["class_idx"].values == idx
        if mask.sum() > 2:
            axes[1].hist(np.clip(u_test[mask], 0, 2), bins=20,
                         alpha=0.5, label=f"{name} (n={mask.sum()})", density=True)
    axes[1].set_xlabel("Normalised epistemic uncertainty (clipped at 2)")
    axes[1].set_ylabel("Density")
    axes[1].set_title("Per-class uncertainty distribution (test)")
    axes[1].legend(fontsize=7, loc="upper right")

    plt.tight_layout()
    out = get_results_dir(DATASET, "comparison") / "uncertainty_distributions.png"
    plt.savefig(str(out), dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
#Per class iou
iou_export = iou_df.round(4)
iou_export.index.name = "Class"
iou_csv = get_results_dir(DATASET, "comparison") / "per_class_iou_table.csv"
iou_export.to_csv(iou_csv)
print(f"Per-class IoU CSV -> {iou_csv}")